# Titian — Record-Level Provenance for Spark SQL

This notebook runs a multi-stage SQL query (join + aggregation) on stock **Spark 4.1**,
then uses **Titian** to trace a suspicious result *backward through both shuffles* to
the exact source records in two different input files — and forward again.

Titian attaches as a library (`spark.sql.extensions`); capture code is fused into
Spark's whole-stage-codegen generated Java. See the VLDB '16 paper *Titian: Data
Provenance Support in Spark* for the design.

In [ ]:
import os, sys, glob

ROOT = os.environ.get("TITIAN_HOME") or os.path.abspath("..")
TITIAN_JAR = os.environ.get("TITIAN_JAR") or sorted(
    glob.glob(f"{ROOT}/target/scala-2.13/titian_*.jar"))[-1]
FASTUTIL_JAR = os.environ.get("FASTUTIL_JAR") or sorted(
    glob.glob(os.path.expanduser("~/Library/Caches/Coursier/**/fastutil-8.5.15.jar"), recursive=True)
    + glob.glob(os.path.expanduser("~/.cache/coursier/**/fastutil-8.5.15.jar"), recursive=True))[-1]
DATA = f"{ROOT}/src/test/resources"
sys.path.insert(0, f"{ROOT}/python")
print("titian jar:", TITIAN_JAR)

In [ ]:
from pyspark.sql import SparkSession
from titian import Titian

spark = (SparkSession.builder
    .master("local[2]")
    .appName("titian-notebook")
    .config("spark.jars", f"{TITIAN_JAR},{FASTUTIL_JAR}")
    .config("spark.sql.extensions", "org.apache.spark.sql.lineage.TitianSQLExtension")
    .config("spark.sql.adaptive.skewJoin.enabled", "false")
    .config("spark.ui.enabled", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

t = Titian(spark)
t.enable_capture()

In [ ]:
# orders: "orderId,customerId,amount" — o8 carries a corrupt amount of 99999
spark.read.schema("oid STRING, cid STRING, amount INT") \
    .csv(f"{DATA}/orders_csv").createOrReplaceTempView("orders")
spark.read.schema("cid STRING, name STRING") \
    .csv(f"{DATA}/customers_csv").createOrReplaceTempView("customers")
spark.read.schema("category STRING, amount INT") \
    .csv(f"{DATA}/sales_parts").createOrReplaceTempView("sales")
spark.table("orders").show()

## A two-stage query with a suspicious result

Revenue per customer. Bob's total is absurd — somewhere in the inputs there is a
corrupt record, but the aggregation has destroyed any direct link to it.

In [ ]:
df = spark.sql("""
    SELECT c.name, SUM(o.amount) AS total
    FROM orders o JOIN customers c ON o.cid = c.cid
    GROUP BY c.name""")
out = t.collect_with_lineage(df)
for row, lineage_id in out:
    print(row, " lineage id:", lineage_id)

totals = {r["name"]: r["total"] for r, _ in out}
assert totals["Bob"] == 100644, totals

## Backward trace: two hops, into both files

`go_back()` walks one capture level at a time: through the aggregation exchange to the
joined-row level, then `go_back(0)` / `go_back(1)` follow the join's two inputs.

In [ ]:
bob_id = next(i for r, i in out if r["name"] == "Bob")

cursor = t.trace(df, [bob_id]).go_back()   # through the aggregation
orders_side = cursor.go_back(0)            # join input 0 = orders
witnesses = orders_side.show(full=True)
for w in witnesses:
    print(w)

amounts = sorted(w["amount"] for w in witnesses)
assert amounts == [190, 205, 250, 99999], amounts
print("\n--> corrupt source record:", max(witnesses, key=lambda w: w["amount"]))

In [ ]:
customers_side = cursor.go_back(1)         # join input 1 = customers
assert [w["name"] for w in customers_side.show()] == ["Bob"]
customers_side.show()

## ROLLUP: grouping-set provenance

The grand-total row depends on *every* input; each category row on exactly its own.

In [ ]:
roll = spark.sql(
    "SELECT category, SUM(amount) AS total FROM sales GROUP BY ROLLUP(category)")
rout = t.collect_with_lineage(roll)
total_id = next(i for r, i in rout if r["category"] is None)
elec_id = next(i for r, i in rout if r["category"] == "electronics")
assert len(t.trace(roll, [total_id]).go_back().ids) == 16
assert len(t.trace(roll, [elec_id]).go_back().ids) == 4
print("rollup provenance OK: total->16 rows, electronics->4 rows")

## Fail-loud coverage

Anything Titian cannot capture faithfully aborts with a clear error — never silent
wrong lineage.

In [ ]:
try:
    spark.sql("SELECT * FROM sales a JOIN sales b ON a.amount > b.amount").collect()
    raise AssertionError("should have failed loudly")
except Exception as e:
    msg = str(e)
    assert "Titian" in msg, msg
    print(msg.splitlines()[0])

In [ ]:
t.release_lineage(df)   # drop this query's lineage blocks
spark.stop()
print("SQL NOTEBOOK OK")